## Libraries

In [ ]:
import os
import re
import sys
from pprint import pprint

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from tqdm.auto import tqdm

sys.path.append(os.path.abspath('..'))
from src.modules.tokenizer import Tokenizer
from src.utils.visualization import plot_scores, t_corr, t_corr_all

load_dotenv(override=True)
tqdm.pandas()
ANNUAL_REPORTS_DIR = os.getenv('ANNUAL_REPORTS_DIR')
GOLD_SUMMARIES_DIR = os.getenv('GOLD_SUMMARIES_DIR')
CANDIDATE_SUMMARIES_DIR = os.getenv('CANDIDATE_SUMMARIES_DIR')
RESULTS_PATH = os.getenv('RESULTS_PATH')
SUMMARY_VER = os.getenv('SUMMARY_VER')

### Annual reports and gold summaries

In [ ]:
rows = []

for language in os.listdir('data/'):
    for dataset in ['training', 'validation']:
        for doc_type in ['annual_reports', 'gold_summaries', 'candidate_summaries']:
            folder_path = f'data/{language}/{dataset}/{doc_type}'
            if not os.path.exists(folder_path):
                continue
            for file in os.listdir(folder_path):
                with open(f'{folder_path}/{file}', 'r', encoding='utf-8') as f:
                    text = f.read()
                    file_name = file.removesuffix(os.getenv('FILE_EXTENSION', '.txt'))
                    
                    match doc_type:
                        case 'annual_reports':
                            doc_id = file_name
                            summary_version = None
                        case 'gold_summaries':
                            try:
                                doc_id = re.findall(r'^(.*)_[^_]+$', file_name)[0]
                                summary_version = re.findall(r'_([^_]+)$', file_name)[0]
                            except:
                                print(file)
                                print(file_name)
                                print(re.findall(r'^(.*)_[^_]+$', file_name))
                        case 'candidate_summaries':
                            try:
                                doc_id = re.findall(r'^\d+', file_name)[0]
                                summary_version = file_name[-3:]
                            except:
                                print(file)
                                print(file_name)
                                print(re.findall(r'^\d+', file_name))
                    rows.append({
                        'doc_id': doc_id,
                        'dataset': dataset,
                        'version': summary_version,
                        'doc_type': doc_type,
                        'language': language,
                        'text': text,
                    })

df = pd.DataFrame(rows)

In [ ]:
df.to_parquet("df.parquet")

In [ ]:
df = pd.read_parquet("df.parquet")

In [ ]:
for language, language_code in zip(['English', 'Greek', 'Spanish'], ['en', 'el', 'es']):
    filter = df['language'] == language
    tokenizer = Tokenizer(lang_code=language_code)
    texts = df.loc[filter, 'text'].to_list()

    char_counts = []
    token_counts = []
    sentence_counts = []

    for doc in tokenizer.nlp.pipe(
        tqdm(texts, desc=f"Processing {language}", unit="doc"),
        batch_size=16,
        n_process=4
        ):
        char_counts.append(len(doc.text))
        token_counts.append(len(doc))
        sentence_counts.append(len([sent.text for sent in doc.sents]))
    
    df.loc[filter, 'char_count'] = char_counts
    df.loc[filter, 'token_count'] = token_counts
    df.loc[filter, 'sent_count'] = sentence_counts

In [ ]:
# Count documents by language and doc_type
counts = (
    df
    .groupby(['language', 'doc_type'])
    .size()
    .unstack(fill_value=0)                                
)

# Plot stacked bar
counts.plot(kind='bar', stacked=True)
plt.title('Number of documents per language (filtered by version)')
plt.ylabel('Document count')
plt.xticks(rotation=0)
plt.legend(title='Document Type')
plt.tight_layout()
plt.show()

In [ ]:
# Filter per language/version as you specified
fdf = df[
    (pd.isna(df['version'])) |
    ((df['language'] == 'English') & (df['version'] == '1')) |
    ((df['language'] == 'Greek') & (df['version'] == '2')) |
    ((df['language'] == 'Spanish') & (df['version'] == 'GS1'))
]

In [ ]:
# Count documents by language and doc_type
counts = (
    fdf
    .groupby(['language', 'doc_type'])
    .size()
    .unstack(fill_value=0)                                
)

# Plot stacked bar
counts.plot(kind='bar', stacked=True)
plt.title('Number of documents per language (filtered by version)')
plt.ylabel('Document count')
plt.xticks(rotation=0)
plt.legend(title='Document Type')
plt.tight_layout()
plt.show()


In [ ]:
fdf['text'].str.len().hist(by=fdf['language'], bins=50, figsize=(12, 8), alpha=0.5)

In [ ]:
from src.modules.tokenizer import Tokenizer
tokenizer = Tokenizer(lang_code='en')
df = df.copy()
df['source_doc_char_count'] = df['source_doc'].apply(lambda x: len(x))
df['candidate_char_count'] = df['candidate'].apply(lambda x: len(x))
df['source_doc_word_count'] = df['source_doc'].apply(lambda x: len(x.split()))
df['candidate_word_count'] = df['candidate'].apply(lambda x: len(x.split()))
df['source_doc_sentence_count'] = df['source_doc'].apply(lambda x: len(re.findall(r'\w+[.!?]', x)))
df['candidate_sentence_count'] = df['candidate'].apply(lambda x: len(re.findall(r'\w+[.!?]', x)))

In [ ]:
for df in dfs:
    df = df.copy()
    df['source_doc_char_count'] = df['source_doc'].apply(lambda x: len(x))
    df['candidate_char_count'] = df['candidate'].apply(lambda x: len(x))
    df['source_doc_word_count'] = df['source_doc'].apply(lambda x: len(x.split()))
    df['candidate_word_count'] = df['candidate'].apply(lambda x: len(x.split()))
    df['source_doc_sentence_count'] = df['source_doc'].apply(lambda x: len(re.findall(r'\w+[.!?]', x)))
    df['candidate_sentence_count'] = df['candidate'].apply(lambda x: len(re.findall(r'\w+[.!?]', x)))